<a href="https://colab.research.google.com/github/amitpaudel21/ARTIFICIAL_INTELLIGENCE_LAB/blob/main/1BF24CS042_LAB6_UNIFICATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re


def parse(s):
    tokens = re.findall(r"[A-Za-z_]\w*|\(|\)|,", s)
    pos = 0
    def term():
        nonlocal pos
        name = tokens[pos]; pos += 1
        if pos < len(tokens) and tokens[pos] == "(":
            pos += 1
            args = [term()]
            while tokens[pos] == ",":
                pos += 1
                args.append(term())
            pos += 1
            return (name, *args)
        return name
    return term()

def is_var(t):
    return isinstance(t, str) and t[0].islower()

def show(t):
    if isinstance(t, str):
        return t
    return f"{t[0]}({', '.join(show(a) for a in t[1:])})"

def substitute(t, theta):
    if is_var(t):
        return substitute(theta[t], theta) if t in theta else t
    if isinstance(t, str):
        return t
    return (t[0], *(substitute(a, theta) for a in t[1:]))

def occurs(v, t, theta):
    t = substitute(t, theta)
    if is_var(t):
        return v == t
    if isinstance(t, str):
        return False
    return any(occurs(v, a, theta) for a in t[1:])

def unify_var(v, t, theta):
    if v in theta:
        return unify(theta[v], t, theta)
    if is_var(t) and t in theta:
        return unify(v, theta[t], theta)
    if occurs(v, t, theta):
        return None
    return {**theta, v: t}

def unify(x, y, theta=None):
    if theta is None:
        theta = {}
    if x == y:
        return theta
    if is_var(x):
        return unify_var(x, y, theta)
    if is_var(y):
        return unify_var(y, x, theta)
    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x) != len(y):
            return None
        for a, b in zip(x[1:], y[1:]):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta
    return None

def run(e1, e2):
    theta = unify(parse(e1), parse(e2))
    print(f"Unify({e1}, {e2})")
    if theta is None:
        print(" -> FAILURE\n")
    else:
        mgu = {k: show(substitute(v, theta)) for k, v in theta.items()}
        print(" -> MGU:", mgu)
        print(" -> Unified:", show(substitute(parse(e1), theta)), "\n")

if __name__ == "__main__":
    run("Eats(x, Apple)", "Eats(Riya, y)") # success
    run("Knows(John, x)", "Knows(x, Elizabeth)") # fail
    run("P(x, h(y))", "P(a, f(z))") # fail
    run("p(f(A), g(y))", "p(x, x)") # fail
    run("P(x, f(y))", "P(Mary, f(Bob))") # success
    run("P(x, f(x))", "P(y, y)") # fail (occurs check)
    run("Q(x, y)", "Q(x)") # fail (arity)


Unify(Eats(x, Apple), Eats(Riya, y))
 -> MGU: {'x': 'Riya', 'y': 'Apple'}
 -> Unified: Eats(Riya, Apple) 

Unify(Knows(John, x), Knows(x, Elizabeth))
 -> FAILURE

Unify(P(x, h(y)), P(a, f(z)))
 -> FAILURE

Unify(p(f(A), g(y)), p(x, x))
 -> FAILURE

Unify(P(x, f(y)), P(Mary, f(Bob)))
 -> MGU: {'x': 'Mary', 'y': 'Bob'}
 -> Unified: P(Mary, f(Bob)) 

Unify(P(x, f(x)), P(y, y))
 -> FAILURE

Unify(Q(x, y), Q(x))
 -> FAILURE

